# Pulpermill: cards and chapters

Run the cells top to bottom (the model takes a few minutes to load), then copy the URL printed by the last cell into the Pulpermill page. There are no datasets to update: every session starts by cloning the code from GitHub.

**Set up once**
- Notebook settings: Internet on, accelerator GPU T4 x2.
- Add-ons, Secrets: `NGROK_AUTHTOKEN` (your ngrok token, required) and `HF_TOKEN` (optional: the model repo is not gated).
- The page is `index.html` in the repo. Open it from your computer (double-click) or from the repo's GitHub Pages address, and paste the URL below into it.

**Changing the code:** push to the repo and start a new session. Re-running the clone cell in a session that already imported the code would keep running the old version.


In [ ]:
# Install
import subprocess, sys
from importlib import metadata


def pip(*packages):
    """pip install, quietly, but a failure stops the notebook here with pip's own output (a "!pip" line that fails lets the cells after it run on)."""
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-2000:] + result.stderr[-3000:])
        raise RuntimeError("pip install failed (its output is above): " + " ".join(packages))


print("installing llama-cpp-python: a minute or two, much longer if there is no ready-made build and it has to be compiled")
pip("llama-cpp-python", "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu121")
print("installing the other packages")
pip("pydantic", "huggingface_hub", "fastapi", "uvicorn", "pyngrok")

# What this session got. (No versions are pinned: this is where a session that behaves differently from the last one can be traced.)
print("\npython", sys.version.split()[0])
for name in ("llama-cpp-python", "pydantic", "huggingface-hub", "fastapi", "uvicorn", "pyngrok"):
    try:
        print(f"{name:18s}", metadata.version(name))
    except metadata.PackageNotFoundError:
        print(f"{name:18s}", "NOT INSTALLED")
try:
    import llama_cpp
except Exception as e:
    raise RuntimeError(f"llama-cpp-python is installed but cannot be loaded ({type(e).__name__}: {e})") from None
if not llama_cpp.llama_supports_gpu_offload():
    print("\nWARNING: this llama-cpp-python build cannot use the GPU. It was probably compiled from the package index instead of "
          "taken from the ready-made CUDA builds, and the model would run on the CPU, far too slowly. Check the versions above.")
else:
    print("GPU offload: supported")


In [ ]:
# Get the code: a fresh clone every session (sessions are not persistent)
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/LacChe/pulpermill"
REPO_REF = None                      # a branch, tag or commit to run instead of the default branch
REPO_DIR = "/kaggle/working/pulpermill"

shutil.rmtree(REPO_DIR, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR] if REPO_REF is None else ["git", "clone", REPO_URL, REPO_DIR], check=True)
if REPO_REF is not None:
    subprocess.run(["git", "-C", REPO_DIR, "checkout", REPO_REF], check=True)
sys.path.insert(0, REPO_DIR)
print("running commit", subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"]).decode().strip())


In [ ]:
# Settings. The defaults are in pulpermill/config.py; change any of them here, before the next cells run.
from pulpermill import config

# config.override(N_CTX=8192, GRAMMAR_MODE="fallback")


In [ ]:
# Load the model (downloads it the first time in a session)
from pulpermill import model

model.load_model()


In [ ]:
# Start the server and the ngrok tunnel. Paste the printed URL into the HTML page.
import json, threading, time, urllib.error, urllib.request
import uvicorn
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
from pulpermill import config
from pulpermill.server import app            # imported after the settings cell: the request limits are built when this loads

# 1. The server. Run again, this cell keeps the server that is already running (a second one could not use the port).
if globals().get("server") is not None and server.started and server.config.app is app:
    print("the server is already running; keeping it")
else:
    server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=config.PORT, log_level="warning"))
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    for _ in range(100):                     # waits for it to be up, for 10 seconds at most
        if server.started:
            break
        if not thread.is_alive():
            raise RuntimeError(f"the server could not start. Is port {config.PORT} used by something else? "
                               "Restart the session, or set another PORT in the settings cell.")
        time.sleep(0.1)
    else:
        raise RuntimeError("the server did not finish starting within 10 seconds")

# 2. Check that it answers and that the model behind it works, before anyone is given the address.
try:
    health = json.load(urllib.request.urlopen(f"http://127.0.0.1:{config.PORT}/health", timeout=30))
except urllib.error.HTTPError as e:         # 503: it is up, but it says it cannot write
    try:
        why = json.loads(e.read().decode("utf-8", "replace")).get("why") or e.reason
    except ValueError:
        why = e.reason
    raise RuntimeError(f"the server is up but not ready: {why}") from None
print("the server answers and the model works:", health["model"])

# 3. The tunnel, now that there is a working server behind it.
ngrok.set_auth_token(UserSecretsClient().get_secret(config.NGROK_SECRET))
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)
url = ngrok.connect(config.PORT).public_url
print("Paste this into the card generator page:\n\n   ", url)
